# ACM sur les données Parcoursup

Ce notebook reprend la logique du TP du dossier `mca/` et l'applique aux données Parcoursup. Il est autonome : il recharge les données, reconstruit les variables utiles, puis déroule l'ACM étape par étape avec les mêmes familles de graphes.

In [1]:
library(tidyverse)
library(FactoMineR)
library(factoextra)
library(ggpubr)

data <- read.csv('Parcoursup.csv', sep = ';', header = TRUE, stringsAsFactors = TRUE, check.names = TRUE)

dim(data)
head(data, 3)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Welcome to factoextra!

Want to learn more? See two factoextra-related books at https://www.datanovia.com/en/product/practical-guide-to-principal-component-methods-in-r/



[1] 14252   118

,Session,Statut.de.l.établissement.de.la.filière.de.formation..public..privé..,Code.UAI.de.l.établissement,Établissement,Code.départemental.de.l.établissement,Département.de.l.établissement,Région.de.l.établissement,Académie.de.l.établissement,Commune.de.l.établissement,Filière.de.formation,⋯,tri,cod_aff_form,Concours.communs.et.banque.d.épreuves,Lien.de.la.formation.sur.la.plateforme.Parcoursup,Taux.d.accès,Part.des.terminales.générales.qui.étaient.en.position.de.recevoir.une.proposition.en.phase.principale,Part.des.terminales.technologiques.qui.étaient.en.position.de.recevoir.une.proposition.en.phase.principale,Part.des.terminales.professionnelles.qui.étaient.en.position.de.recevoir.une.proposition.en.phase.principale,etablissement_id_paysage,composante_id_paysage
,<int>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,⋯,<fct>,<int>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<lgl>,<lgl>
1,2025,Public,0692185A,INSTITUT DES SCIENCES ET TECHNIQUES DE LA READAPTATION UNIVERSITE LYON 1,69,Rhône,Auvergne-Rhône-Alpes,Lyon,Lyon 8e Arrondissement,Certificat de capacité d'Orthoptiste,⋯,3_Autres formations,28087,Aix-Marseille Université - Site de Marseille Timone,https://dossierappel.parcoursup.fr/Candidats/public/fiches/afficherFicheFormation?g_ta_cod=28087&typeBac=0&originePc=0,13,93,7,1,NA,NA
2,2025,Public,0931827F,Université Paris 8,93,Seine-Saint-Denis,Ile-de-France,Créteil,Saint-Denis,Licence - Langues étrangères appliquées - Parcours Anglais / Italien,⋯,1_universités,28100,,https://dossierappel.parcoursup.fr/Candidats/public/fiches/afficherFicheFormation?g_ta_cod=28100&typeBac=0&originePc=0,98,57,24,19,NA,NA
3,2025,Public,0421573G,IFSI du CH de Roanne,42,Loire,Auvergne-Rhône-Alpes,Lyon,Roanne,D.E Infirmier,⋯,3_Autres formations,28144,,https://dossierappel.parcoursup.fr/Candidats/public/fiches/afficherFicheFormation?g_ta_cod=28144&typeBac=0&originePc=0,40,56,37,7,NA,NA


## Préparation des données

On reprend ici la préparation utile de `Data.ipynb` pour disposer d'un notebook ACM exécutable seul.

In [2]:
cols_to_drop <- c(
  'Session',
  'Code.UAI.de.l.établissement',
  'Code.départemental.de.l.établissement',
  'Département.de.l.établissement',
  'Coordonnées.GPS.de.la.formation',
  'Commune.de.l.établissement',
  'Filière.de.formation.détaillée',
  'Filière.de.formation.détaillée.bis',
  'Filière.de.formation.très.détaillée',
  'Dont.effectif.des.candidats.ayant.postulé.en.internat',
  'Effectif.total.des.candidats.classés.par.l.établissement.en.phase.principale',
  'Effectif.des.candidats.classés.par.l.établissement.en.phase.complémentaire',
  'Effectif.des.candidats.classés.par.l.établissement.en.internat..CPGE.',
  'Effectif.des.candidats.classés.par.l.établissement.hors.internat..CPGE.',
  'Concours.communs.et.banque.d.épreuves',
  'cod_aff_form',
  'Lien.de.la.formation.sur.la.plateforme.Parcoursup',
  'etablissement_id_paysage',
  'composante_id_paysage'
)

num_cols <- c(
  'Capacité.de.l.établissement.par.formation',
  'Effectif.total.des.candidats.en.phase.principale',
  'Effectif.total.des.candidats.en.phase.complémentaire',
  'Effectif.des.admis.en.phase.principale',
  'Effectif.des.admis.en.phase.complémentaire',
  'Effectif.total.des.candidats.ayant.accepté.la.proposition.de.l.établissement..admis.',
  'Dont.effectif.des.candidates.admises',
  'Effectif.des.admis.néo.bacheliers',
  'Dont.effectif.des.admis.néo.bacheliers.sans.mention.au.bac',
  'Dont.effectif.des.admis.néo.bacheliers.avec.mention.Assez.Bien.au.bac',
  'Dont.effectif.des.admis.néo.bacheliers.avec.mention.Bien.au.bac',
  'Dont.effectif.des.admis.néo.bacheliers.avec.mention.Très.Bien.au.bac',
  'Dont.effectif.des.admis.néo.bacheliers.avec.mention.Très.Bien.avec.félicitations.au.bac'
)

data <- data %>%
  select(-any_of(cols_to_drop)) %>%
  mutate(across(any_of(num_cols), ~ suppressWarnings(as.numeric(.x)))) %>%
  mutate(
    Effectif.total.candidats.2phases = coalesce(Effectif.total.des.candidats.en.phase.principale, 0) +
      coalesce(Effectif.total.des.candidats.en.phase.complémentaire, 0),
    Effectif.total.admis.2phases = coalesce(Effectif.des.admis.en.phase.principale, 0) +
      coalesce(Effectif.des.admis.en.phase.complémentaire, 0),
    pct_filles_admises = if_else(
      Effectif.total.des.candidats.ayant.accepté.la.proposition.de.l.établissement..admis. > 0,
      Dont.effectif.des.candidates.admises / Effectif.total.des.candidats.ayant.accepté.la.proposition.de.l.établissement..admis.,
      NA_real_
    ),
    pct_garcons_admis = if_else(!is.na(pct_filles_admises), 1 - pct_filles_admises, NA_real_),
    part_mention_ab = if_else(Effectif.des.admis.néo.bacheliers > 0,
      Dont.effectif.des.admis.néo.bacheliers.avec.mention.Assez.Bien.au.bac / Effectif.des.admis.néo.bacheliers, NA_real_),
    part_mention_b = if_else(Effectif.des.admis.néo.bacheliers > 0,
      Dont.effectif.des.admis.néo.bacheliers.avec.mention.Bien.au.bac / Effectif.des.admis.néo.bacheliers, NA_real_),
    part_mention_tb = if_else(Effectif.des.admis.néo.bacheliers > 0,
      Dont.effectif.des.admis.néo.bacheliers.avec.mention.Très.Bien.au.bac / Effectif.des.admis.néo.bacheliers, NA_real_),
    part_mention_tb_fel = if_else(Effectif.des.admis.néo.bacheliers > 0,
      Dont.effectif.des.admis.néo.bacheliers.avec.mention.Très.Bien.avec.félicitations.au.bac / Effectif.des.admis.néo.bacheliers, NA_real_)
  ) %>%
  rename(Statut.Etablissement = "Statut.de.l.établissement.de.la.filière.de.formation..public..privé..") %>%
  mutate(
    Statut.Etablissement = case_when(
      grepl('^Public', Statut.Etablissement) ~ 'Public',
      grepl('^Privé', Statut.Etablissement) ~ 'Privé',
      TRUE ~ as.character(Statut.Etablissement)
    ),
    total_candidats = Effectif.des.candidats.néo.bacheliers.généraux.en.phase.principale +
      Effectif.des.candidats.néo.bacheliers.technologiques.en.phase.principale +
      Effectif.des.candidats.néo.bacheliers.professionnels.en.phase.principale,
    total_boursiers_candidats = Dont.effectif.des.candidats.boursiers.néo.bacheliers.généraux.en.phase.principale +
      Dont.effectif.des.candidats.boursiers.néo.bacheliers.technologiques.en.phase.principale +
      Dont.effectif.des.candidats.boursiers.néo.bacheliers.professionnels.en.phase.principale,
    Pourcentage.boursiers.candidats = (total_boursiers_candidats / total_candidats) * 100,
    Pourcentage.boursiers.admis = (Dont.effectif.des.admis.boursiers.néo.bacheliers / Effectif.des.admis.néo.bacheliers) * 100,
    taux_acces_num = suppressWarnings(as.numeric(gsub(',', '.', gsub('%', '', Taux.d.accès)))) / 100
  ) %>%
  select(-total_candidats, -total_boursiers_candidats)

data_features_core <- data %>%
  mutate(
    pression_candidature = if_else(Capacité.de.l.établissement.par.formation > 0,
      Effectif.total.candidats.2phases / Capacité.de.l.établissement.par.formation, NA_real_),
    taux_admission = if_else(Effectif.total.candidats.2phases > 0,
      Effectif.total.admis.2phases / Effectif.total.candidats.2phases, NA_real_),
    taux_remplissage = if_else(Capacité.de.l.établissement.par.formation > 0,
      Effectif.total.admis.2phases / Capacité.de.l.établissement.par.formation, NA_real_),
    part_mention_haute = part_mention_tb + part_mention_tb_fel,
    score_mention = 1 * coalesce(part_mention_ab, 0) +
      2 * coalesce(part_mention_b, 0) +
      3 * coalesce(part_mention_tb, 0) +
      4 * coalesce(part_mention_tb_fel, 0),
    log_candidats = log1p(Effectif.total.candidats.2phases),
    log_admis = log1p(Effectif.total.admis.2phases),
    log_capacite = log1p(Capacité.de.l.établissement.par.formation),
    pression_q = ntile(pression_candidature, 4),
    taux_acces_q = ntile(taux_acces_num, 4)
  ) %>%
  select(
    Établissement, Statut.Etablissement, Filière.de.formation, Filière.de.formation.très.agrégée,
    Région.de.l.établissement, Académie.de.l.établissement, Sélectivité,
    Capacité.de.l.établissement.par.formation, Effectif.total.admis.2phases, Effectif.total.candidats.2phases,
    Pourcentage.boursiers.candidats, Pourcentage.boursiers.admis,
    pct_filles_admises, pct_garcons_admis,
    pression_candidature, taux_admission, taux_remplissage, taux_acces_num,
    part_mention_ab, part_mention_b, part_mention_tb, part_mention_tb_fel, part_mention_haute, score_mention,
    X..d.admis.néo.bacheliers.issus.de.la.même.académie..Paris.Créteil.Versailles.réunies.,
    X..d.admis.néo.bacheliers.généraux,
    X..d.admis.néo.bacheliers.technologiques,
    X..d.admis.néo.bacheliers.professionnels,
    log_candidats, log_admis, log_capacite,
    pression_q, taux_acces_q
  ) %>%
  mutate(across(where(is.numeric), ~ ifelse(is.na(.), median(., na.rm = TRUE), .)))

glimpse(data_features_core)

Rows: 14,252
Columns: 33
$ Établissement                                                                          <fct> …
$ Statut.Etablissement                                                                   <chr> …
$ Filière.de.formation                                                                   <fct> …
$ Filière.de.formation.très.agrégée                                                      <fct> …
$ Région.de.l.établissement                                                              <fct> …
$ Académie.de.l.établissement                                                            <fct> …
$ Sélectivité                                                                            <fct> …
$ Capacité.de.l.établissement.par.formation                                              <dbl> …
$ Effectif.total.admis.2phases                                                           <dbl> …
$ Effectif.total.candidats.2phases                                                       <dbl> …
$ Pou

## Choix des variables pour l'ACM

Comme dans le TP, on sépare variables actives et variables supplémentaires. On garde comme variables actives des variables qualitatives structurantes, et on construit des variables supplémentaires en classes à partir de variables numériques.

In [5]:
active_vars <- c(
  'Sélectivité',
  'Statut.Etablissement',
  'Filière.de.formation.très.agrégée',
  'Région.de.l.établissement'
)

data_mca <- data_features_core %>%
  mutate(
    pression_q = factor(pression_q, labels = c('pression faible', 'pression modérée', 'pression élevée', 'pression très élevée')),
    taux_acces_q = factor(taux_acces_q, labels = c('accès faible', 'accès modéré', 'accès élevé', 'accès très élevé')),
    boursiers_q = factor(ntile(Pourcentage.boursiers.admis, 4), labels = c('boursiers faibles', 'boursiers modérés', 'boursiers élevés', 'boursiers très élevés')),
    filles_q = factor(ntile(pct_filles_admises, 4), labels = c('part filles faible', 'part filles modérée', 'part filles élevée', 'part filles très élevée')),
    across(all_of(active_vars), as.factor)
  ) %>%
  select(all_of(active_vars), pression_q, taux_acces_q, boursiers_q, filles_q) %>%
  drop_na()

summary(data_mca)
head(data_mca)

ERROR: [1m[33mError[39m in `mutate()`:[22m
[1m[22m[36mℹ[39m In argument: `pression_q = factor(...)`.
[1mCaused by error in `factor()`:[22m
[33m![39m invalid 'labels'; length 4 should be 1 or 5


## Fréquences des modalités

On commence comme dans le TP par regarder la distribution de chaque variable.

In [4]:
# Variables actives
par(mfrow = c(2, 2))
for (i in 1:length(active_vars)) {
  plot(data_mca[, i], main = colnames(data_mca)[i], ylab = 'Count', col = 'steelblue', las = 2)
}

# Variables supplémentaires
par(mfrow = c(2, 2))
for (i in (length(active_vars) + 1):ncol(data_mca)) {
  plot(data_mca[, i], main = colnames(data_mca)[i], ylab = 'Count', col = 'steelblue', las = 2)
}

par(mfrow = c(1, 1))

ERROR: Error: object 'data_mca' not found


## Calcul de l'ACM

Les 4 premières variables sont actives, les 4 suivantes sont supplémentaires.

In [ ]:
res.mca <- MCA(data_mca, quali.sup = 5:8, graph = FALSE)
res.mca

## Inertie des axes

In [ ]:
head(res.mca$eig)

fviz_screeplot(res.mca, addlabels = TRUE)
fviz_screeplot(res.mca, addlabels = TRUE, ncp = min(8, nrow(res.mca$eig)))

## Qualité de représentation des modalités

In [ ]:
head(res.mca$var$cos2)

i <- c(1, 2)
idx <- which.max(rowSums(res.mca$var$cos2[, i]))
best_modality <- rownames(res.mca$var$cos2)[idx]
print(paste('La modalité la mieux représentée sur le plan', i[1], '-', i[2], 'est', best_modality))

idx <- which.min(rowSums(res.mca$var$cos2[, i]))
worst_modality <- rownames(res.mca$var$cos2)[idx]
print(paste('La modalité la moins bien représentée sur le plan', i[1], '-', i[2], 'est', worst_modality))

i <- c(3, 4)
idx <- which.max(rowSums(res.mca$var$cos2[, i]))
best_modality <- rownames(res.mca$var$cos2)[idx]
print(paste('La modalité la mieux représentée sur le plan', i[1], '-', i[2], 'est', best_modality))

idx <- which.min(rowSums(res.mca$var$cos2[, i]))
worst_modality <- rownames(res.mca$var$cos2)[idx]
print(paste('La modalité la moins bien représentée sur le plan', i[1], '-', i[2], 'est', worst_modality))

## Représentation des modalités

In [ ]:
fviz_mca_var(res.mca, col.var = 'cos2', gradient.cols = c('blue', 'yellow', 'red'), repel = TRUE)
fviz_mca_var(res.mca, axes = c(3, 4), col.var = 'cos2', gradient.cols = c('blue', 'yellow', 'red'), repel = TRUE)

fviz_cos2(res.mca, choice = 'var', axes = 1:2)
fviz_cos2(res.mca, choice = 'var', axes = 3:4)

## Contributions des modalités aux axes

In [ ]:
fviz_contrib(res.mca, choice = 'var', axes = 1, top = 20)
fviz_contrib(res.mca, choice = 'var', axes = 2, top = 20)
fviz_contrib(res.mca, choice = 'var', axes = 3, top = 20)
fviz_contrib(res.mca, choice = 'var', axes = 4, top = 20)

## Graphique groupé des modalités

In [ ]:
var_groups <- sub('=.*', '', rownames(res.mca$var$coord))
var_groups <- sub(':.*', '', var_groups)

fviz_mca_var(res.mca, col.var = var_groups, title = 'Graph of the Parcoursup modalities', repel = TRUE)
fviz_mca_var(res.mca, axes = c(3, 4), col.var = var_groups, title = 'Graph of the Parcoursup modalities (Dim 3-4)', repel = TRUE)

## Représentation des individus

In [ ]:
fviz_mca_ind(res.mca, col.ind = 'cos2', gradient.cols = c('blue', 'yellow', 'red'))
fviz_mca_ind(res.mca, axes = c(3, 4), col.ind = 'cos2', gradient.cols = c('blue', 'yellow', 'red'))

fviz_cos2(res.mca, choice = 'ind', axes = 1:2)
fviz_cos2(res.mca, choice = 'ind', axes = 3:4)

## Individus groupés par modalité

In [ ]:
fviz_mca_ind(res.mca, label = 'none', habillage = 'pression_q', palette = c('steelblue', 'orange', 'darkgreen', 'firebrick'))
fviz_mca_ind(res.mca, label = 'none', habillage = 'taux_acces_q', palette = c('steelblue', 'orange', 'darkgreen', 'firebrick'))
fviz_mca_ind(res.mca, label = 'none', habillage = 'filles_q', palette = c('steelblue', 'orange', 'darkgreen', 'firebrick'))

fviz_ellipses(res.mca, c('pression_q', 'taux_acces_q', 'filles_q'), geom = 'point')

## Biplot

In [ ]:
fviz_mca_biplot(res.mca)
fviz_mca_biplot(res.mca, axes = c(3, 4))

## Corrélation des variables avec les axes

In [ ]:
fviz_mca_var(res.mca, choice = 'mca.cor', repel = TRUE)
fviz_mca_var(res.mca, axes = c(3, 4), choice = 'mca.cor', repel = TRUE)

## Qualité de représentation des variables supplémentaires

In [ ]:
df_sup_12 <- data.frame(quality = rowSums(res.mca$quali.sup$cos2[, c(1, 2)])) %>%
  rownames_to_column('variable')

ggplot(df_sup_12, aes(x = reorder(variable, quality), y = quality)) +
  geom_col(fill = 'steelblue') +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = 'Qualité de représentation des variables supplémentaires sur le plan 1-2', x = NULL, y = 'Qualité')

df_sup_34 <- data.frame(quality = rowSums(res.mca$quali.sup$cos2[, c(3, 4)])) %>%
  rownames_to_column('variable')

ggplot(df_sup_34, aes(x = reorder(variable, quality), y = quality)) +
  geom_col(fill = 'steelblue') +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = 'Qualité de représentation des variables supplémentaires sur le plan 3-4', x = NULL, y = 'Qualité')